[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn.functional as F

In [26]:
# ✏️ YOUR IMPLEMENTATION HERE
def phi(X: torch.tensor) -> torch.tensor:
  return torch.where(X >= 0, X, torch.exp(X) - 1) + 1

def linear_attention(Q, K, V):
    kv = phi(K.mT) @ V # [B, D_k, S] @ [B, S, D_v] = [B, D_k, D_v]
    print(kv.shape)
    phiQ = phi(Q) # [B, S, D_k]
    print(phiQ.shape)
    mulx = (phiQ @ kv)
    print(mulx.shape)
    # Note the transformation. Might also want to learn about linear attention more
    return mulx / (phiQ @ torch.sum(phi(K), dim=-2).unsqueeze(-1))

In [27]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

torch.Size([1, 16, 32])
torch.Size([1, 8, 16])
torch.Size([1, 8, 32])
Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [28]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 16, 32])
torch.Size([2, 8, 16])
torch.Size([2, 8, 32])
  ✅ [1/4] Output shape (1.4ms)
torch.Size([2, 8, 8])
torch.Size([2, 16, 8])
torch.Size([2, 16, 8])
  ✅ [2/4] No NaN or Inf (18.8ms)
torch.Size([1, 8, 8])
torch.Size([1, 4, 8])
torch.Size([1, 4, 8])
  ✅ [3/4] Gradient flow (28.1ms)
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 64, 64])
torch.Size([1, 2048, 64])
torch.Size([1, 2048, 64])